In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
from pathlib import Path
import math
import calendar


## Data loading

### Population

In [6]:
# Define paths to the storm events CSV files
data_dir = Path('../datasets\\')
files = [
    data_dir / 'cc-est2024-alldata.csv'
]

# Load datasets
print("Loading datasets...")
df_list = []
for file in files:
    # print(f"  Loading {file.name}...")
    df = pd.read_csv(file, encoding='latin1')
    # print(f"    Shape: {df.shape}")
    df_list.append(df)

# Concatenate the datasets
df = pd.concat(df_list, ignore_index=True)
print(f"\nCombined dataset shape: {df.shape}")

Loading datasets...

Combined dataset shape: (358416, 80)


In [41]:
df.head(20)

,SUMLEV,STATE,COUNTY,STNAME,CTYNAME,YEAR,AGEGRP,TOT_POP,TOT_MALE,TOT_FEMALE,...,HWAC_MALE,HWAC_FEMALE,HBAC_MALE,HBAC_FEMALE,HIAC_MALE,HIAC_FEMALE,HAAC_MALE,HAAC_FEMALE,HNAC_MALE,HNAC_FEMALE
0,50,1,1,Alabama,Autauga County,1,0,58800,28693,30107,...,963,844,132,116,42,33,22,25,19,10
1,50,1,1,Alabama,Autauga County,1,1,3491,1818,1673,...,93,60,17,11,3,0,11,1,3,0
2,50,1,1,Alabama,Autauga County,1,2,3663,1875,1788,...,92,77,6,9,9,4,0,1,1,2
3,50,1,1,Alabama,Autauga County,1,3,4189,2152,2037,...,94,93,12,13,1,3,2,2,3,1
4,50,1,1,Alabama,Autauga County,1,4,3876,1960,1916,...,79,79,9,11,5,3,2,4,3,2
5,50,1,1,Alabama,Autauga County,1,5,3240,1656,1584,...,63,61,7,7,4,1,2,0,1,0
6,50,1,1,Alabama,Autauga County,1,6,3885,1911,1974,...,91,45,4,6,2,1,3,1,4,0
7,50,1,1,Alabama,Autauga County,1,7,3788,1826,1962,...,73,72,9,4,4,2,1,0,1,1
8,50,1,1,Alabama,Autauga County,1,8,3970,1946,2024,...,116,75,13,7,1,3,0,6,2,3
9,50,1,1,Alabama,Autauga County,1,9,3809,1811,1998,...,63,73,7,6,1,5,0,2,1,0


In [32]:
useful_columns = [
    'TOT_MALE', 'TOT_FEMALE', 'WA_MALE', 'WA_FEMALE', 'BA_MALE',
    'BA_FEMALE', 'IA_MALE', 'IA_FEMALE', 'AA_MALE', 'AA_FEMALE', 'NA_MALE',
    'NA_FEMALE', 'TOM_MALE', 'TOM_FEMALE', 'WAC_MALE', 'WAC_FEMALE',
    'BAC_MALE', 'BAC_FEMALE', 'IAC_MALE', 'IAC_FEMALE', 'AAC_MALE',
    'AAC_FEMALE', 'NAC_MALE', 'NAC_FEMALE', 'NH_MALE', 'NH_FEMALE',
    'NHWA_MALE', 'NHWA_FEMALE', 'NHBA_MALE', 'NHBA_FEMALE', 'NHIA_MALE',
    'NHIA_FEMALE', 'NHAA_MALE', 'NHAA_FEMALE', 'NHNA_MALE', 'NHNA_FEMALE',
    'NHTOM_MALE', 'NHTOM_FEMALE', 'NHWAC_MALE', 'NHWAC_FEMALE',
    'NHBAC_MALE', 'NHBAC_FEMALE', 'NHIAC_MALE', 'NHIAC_FEMALE',
    'NHAAC_MALE', 'NHAAC_FEMALE', 'NHNAC_MALE', 'NHNAC_FEMALE', 'H_MALE',
    'H_FEMALE', 'HWA_MALE', 'HWA_FEMALE', 'HBA_MALE', 'HBA_FEMALE',
    'HIA_MALE', 'HIA_FEMALE', 'HAA_MALE', 'HAA_FEMALE', 'HNA_MALE',
    'HNA_FEMALE', 'HTOM_MALE', 'HTOM_FEMALE', 'HWAC_MALE', 'HWAC_FEMALE',
    'HBAC_MALE', 'HBAC_FEMALE', 'HIAC_MALE', 'HIAC_FEMALE', 'HAAC_MALE',
    'HAAC_FEMALE', 'HNAC_MALE', 'HNAC_FEMALE'
]
df0 = df.drop(columns=useful_columns, inplace=False)
df0.columns

Index(['SUMLEV', 'STATE', 'COUNTY', 'STNAME', 'CTYNAME', 'YEAR', 'AGEGRP',
       'TOT_POP'],
      dtype='object')

In [33]:
df['COUNTY'].nunique()

329

In [15]:
df['COUNTY'].value_counts()

COUNTY
1      5472
5      5472
3      5472
9      5358
7      5244
       ... 
810     114
820     114
830     114
840     114
78      114
Name: count, Length: 329, dtype: int64

### County surface

In [34]:
url_county = data_dir / 'county_landmass.csv'
cs_df = pd.read_csv(url_county)

In [36]:
cs_df.head()

,FIPS,FIPS_state,FIPS_county,state_abbrev,state,county,sq_mi,land_sq_mi,water_sq_mi
0,1001,1,1,AL,Alabama,Autauga,604.45,595.97,8.48
1,1003,1,3,AL,Alabama,Baldwin,2026.93,1596.35,430.58
2,1005,1,5,AL,Alabama,Barbour,904.52,884.90,19.61
3,1007,1,7,AL,Alabama,Bibb,626.16,623.03,3.14
4,1009,1,9,AL,Alabama,Blount,650.60,645.59,5.02


In [37]:
cs_df['FIPS_county'].nunique()

326

### House units

In [30]:
url_housing = data_dir / 'NST-EST2024-HU.xlsx'
column = ['GEOGRAPHIC_AREA', 'ESTIMATES_BASE', 'HU_2020', 'HU_2021', 'HU_2022', 'HU_2023', 'HU_2024']
hu_df = pd.read_excel(url_housing, skiprows=9, names=column, header=None)

In [31]:
hu_df.head(15)

,GEOGRAPHIC_AREA,ESTIMATES_BASE,HU_2020,HU_2021,HU_2022,HU_2023,HU_2024
0,.Alabama,2288337.0,2292694.0,2313291.0,2338998.0,2359765.0,2381817.0
1,.Alaska,326199.0,326593.0,327863.0,329245.0,329677.0,330463.0
2,.Arizona,3081997.0,3092927.0,3139761.0,3189962.0,3243011.0,3299651.0
3,.Arkansas,1365266.0,1368214.0,1380505.0,1395318.0,1408912.0,1421037.0
4,.California,14392141.0,14417494.0,14524995.0,14642752.0,14762262.0,14877904.0
5,.Colorado,2491404.0,2500752.0,2540442.0,2591143.0,2639267.0,2676415.0
6,.Connecticut,1530193.0,1531476.0,1536314.0,1540585.0,1546931.0,1554121.0
7,.Delaware,448735.0,450131.0,457806.0,465556.0,471318.0,476405.0
8,.District of Columbia,350365.0,351429.0,357360.0,360735.0,367036.0,368736.0
9,.Florida,9865359.0,9900384.0,10051531.0,10252120.0,10448447.0,10629918.0
